# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("=== Dataset Overview ===")
print(f"Title: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"URL: {croissant_url}")
print(f"Version: {metadata.get('version', 'N/A')}")
print(f"Published: {metadata.get('datePublished', 'N/A')}")
print(f"License: {metadata.get('license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Record sets, fields, and columns are uniquely identified using their `@id` values for accurate referencing.

In [ ]:
# Get available record sets from metadata
record_sets = metadata.get('recordSet', [])
# If empty, try alternative access (for older schemas)
if not record_sets and hasattr(dataset.metadata, 'recordSet'):
    record_sets = dataset.metadata.recordSet

# Show IDs and basic info for each record set
print("=== Record Sets Overview ===")
if not record_sets:
    print("No record sets found.")
else:
    for rs in record_sets:
        if isinstance(rs, dict):
            rid = rs.get('@id', 'N/A')
            rname = rs.get('name','')
        else:
            rid = rs
            rname = ''
        print(f"Record Set @id: {rid} | Name: {rname}")

    # List fields and columns per record set
    for rs in record_sets:
        rs_id = rs['@id'] if isinstance(rs, dict) else rs
        print(f"\n-- Fields for Record Set @id: {rs_id} --")
        # Retrieve fields (if available)
        rs_obj = dataset.record_set(rs_id)
        if rs_obj and hasattr(rs_obj, 'field'):
            for field in rs_obj.field:
                field_id = field['@id'] if isinstance(field, dict) else field
                field_name = field.get('name','') if isinstance(field, dict) else ''
                print(f"Field @id: {field_id} | Name: {field_name}")
                # Show columns (if present)
                columns = field.get('column',[]) if isinstance(field, dict) else []
                if columns:
                    for col in columns:
                        col_id = col['@id'] if isinstance(col, dict) else col
                        col_name = col.get('name','') if isinstance(col, dict) else ''
                        print(f"    Column @id: {col_id} | Name: {col_name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, all entities are referenced strictly by their `@id`s.

In [ ]:
# Extract data from each record set (using @id)
from collections import OrderedDict

# Collect all record set IDs
record_set_ids = []
for rs in record_sets:
    if isinstance(rs, dict):
        record_set_ids.append(rs['@id'])
    else:
        record_set_ids.append(rs)

dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

if record_set_ids:
    selected_rs_id = record_set_ids[0]
    print(f"Available columns for record set '@id': {selected_rs_id}")
    print(dataframes[selected_rs_id].columns.tolist())
    print("Preview of first 5 records:")
    display(dataframes[selected_rs_id].head())
else:
    print("No record sets to extract.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All field references are done via their `@id` values.

In [ ]:
# Example: Filter records by GAD-7 score (assuming a column 'gad7_score' exists; replace with actual @id)

# Find a numeric field candidate from available columns
numeric_field_id = None
possible_numeric_fields = ['gad7_score', 'phq9_score', 'psq_score', 'age']
df = dataframes[selected_rs_id]
for col in df.columns:
    for candidate in possible_numeric_fields:
        if candidate.lower() in col.lower():
            numeric_field_id = col
            break
    if numeric_field_id:
        break

if numeric_field_id:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a demographic field, e.g. 'village' (find its @id dynamically)
    group_field_id = None
    possible_group_fields = ['village', 'level_of_education', 'marital_status', 'gender']
    for col in df.columns:
        for candidate in possible_group_fields:
            if candidate.lower() in col.lower():
                group_field_id = col
                break
        if group_field_id:
            break
    if group_field_id:
        grouped_df = (
            filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        )
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No numeric field identified for exploratory analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

All visualizations reference fields using their actual column or `@id` names.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution if available
if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Visualize relationship between numeric and group field, if group field available
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Dataset metadata and record sets were loaded using `mlcroissant`, referencing entities by their `@id`.
- Available fields and columns were explored, and data were extracted for analysis.
- Numeric mental health scores (e.g., GAD-7) were filtered and normalized, and grouped analysis was conducted by demographic attributes (e.g., village, gender).
- Visualizations showcased distributions and relationships, enabling insights into mental health survey data for Kilifi County.

For further analysis, consider more advanced modeling, cross-record set joins, and deeper examination of survey response quality and missing data patterns.